# sqrt-eps-stabilize — faded example 3: Identify the correct eps placement from two candidate expressions

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `sqrt-eps-stabilize`. Running the beacon reports progress on the `Numerical: sqrt-eps stabilization` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numerical: sqrt-eps stabilization` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`sqrt-eps-stabilize`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "sqrt-eps-stabilize"
DD_SUBTOPIC = "Numerical: sqrt-eps stabilization"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

Two expressions look almost identical but behave very differently near zero: `sqrt(v + eps)` and `sqrt(v) + eps`. The first is stable — its gradient is bounded by `1/(2*sqrt(eps))`. The second is unstable — `sqrt(v)` has infinite gradient at v=0. When you see `sqrt(...)` in an optimizer or normalization layer, the eps must always be inside the argument to the sqrt.

## Faded exercise 3

Given a vector v containing zeros, compute both candidate denominators and verify which one keeps finite gradients when v=0.

1. Compute `denom_inside = sqrt(v + eps)` — the correct form.
2. Compute `denom_outside = sqrt(v) + eps` — the incorrect form.
3. Compute gradients of sum of each through autograd.

The blank step is computing `denom_inside` with eps placed correctly inside the sqrt.

**Fill in:** Compute denom_inside as the square root of (v + eps), keeping eps inside the sqrt argument.

In [ ]:
import torch as t

t.manual_seed(0)

eps = 1e-8
v_vals = t.tensor([0.0, 1e-7, 0.01, 1.0])

# Correct: eps inside
v_in = v_vals.clone().detach().requires_grad_(True)
denom_inside = None  # TODO: Compute denom_inside as the square root of (v + eps), keeping eps inside the sqrt argument.
denom_inside.sum().backward()
grad_inside = v_in.grad.clone()

# Incorrect: eps outside
v_out = v_vals.clone().detach().requires_grad_(True)
denom_outside = t.sqrt(v_out) + eps
denom_outside.sum().backward()
grad_outside = v_out.grad.clone()

print('grad_inside  (at v=0):', grad_inside[0].item())
print('grad_outside (at v=0):', grad_outside[0].item())  # inf
print('inside finite:', t.isfinite(grad_inside).all().item())
print('outside finite:', t.isfinite(grad_outside).all().item())


def _test():
    import torch as t
    import math

    eps = 1e-8
    v_vals = t.tensor([0.0, 1e-7, 0.01, 1.0])
    v_ref = v_vals.clone().detach().requires_grad_(True)
    ref = t.sqrt(v_ref + eps)
    ref.sum().backward()
    expected_grad = v_ref.grad.clone()

    assert t.allclose(denom_inside, t.sqrt(v_vals + eps), atol=1e-7), 'denom_inside wrong'
    assert t.allclose(grad_inside, expected_grad, atol=1e-6), 'grad_inside mismatch'
    assert t.isfinite(grad_inside).all(), 'inside gradients should all be finite'
    # at v=0, inside grad = 1/(2*sqrt(eps))
    expected_at_zero = 1.0 / (2.0 * math.sqrt(eps))
    assert abs(grad_inside[0].item() - expected_at_zero) < 1.0, f'at v=0: {grad_inside[0].item()}'


try:
    _test()
    _dd_passed.add('faded3')
    print('[Delta Drills] faded3 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded3'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t

t.manual_seed(0)

eps = 1e-8
v_vals = t.tensor([0.0, 1e-7, 0.01, 1.0])

v_in = v_vals.clone().detach().requires_grad_(True)
denom_inside = t.sqrt(v_in + eps)
denom_inside.sum().backward()
grad_inside = v_in.grad.clone()

v_out = v_vals.clone().detach().requires_grad_(True)
denom_outside = t.sqrt(v_out) + eps
denom_outside.sum().backward()
grad_outside = v_out.grad.clone()

print('grad_inside  (at v=0):', grad_inside[0].item())
print('grad_outside (at v=0):', grad_outside[0].item())
print('inside finite:', t.isfinite(grad_inside).all().item())
print('outside finite:', t.isfinite(grad_outside).all().item())
```
</details>